# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haritharamadass/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)



## 1. Ranked actions + reason codes

### Ranked action queue

The purpose of this playbook is to turn the validated model ranking into a practical queue for human content review.

The Logistic Regression model ranks content by estimated risk of declining in the following month. In ML-08, the model achieved Precision@20 of 0.800 on the grouped holdout, compared with 0.700 for the rule baseline. In ML-09, 5-fold grouped validation produced a mean Precision@20 of 0.71.

Because performance varies across unseen client groups and the model produces false positives, the score is used only to decide what should be reviewed first. It does not prove that a page will decline.

I convert the highest-ranked items into four review archetypes:

- `HIGH_VISIBILITY_LOW_CTR` → review title, snippet, search intent, relevance, and freshness.
- `LOW_CTR_FOR_POSITION` → review CTR-related elements such as title, snippet, and intent alignment.
- `HIGH_VISIBILITY_RISK` → inspect freshness, relevance, competition, and recent search-performance changes.
- `GENERAL_RISK_WATCHLIST` → perform a broader manual diagnostic review before making changes.

Freshness is treated as a review hypothesis rather than a guaranteed fix. The earlier FlyRank freshness finding was observational, so refreshing content should not be described as causing an improvement.

Every recommended action requires human review before any content is changed or published.

In [1]:
# ============================================================
# ML-10 — Section 1
# Ranked actions + reason codes
# Rebuild the validated March -> April ranking and convert
# the top-ranked items into a human-review action queue.
# ============================================================

import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

FEATURES = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
]

TARGET = "declined_next_month"
GROUP = "client_hash_id"


# ------------------------------------------------------------
# 1. Connect to the FlyRank warehouse
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face token loaded successfully.")


# ------------------------------------------------------------
# 2. Rebuild the same March -> April feature frame
#    used in the validated model
# ------------------------------------------------------------

MARCH = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

APRIL = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

feature_frame = con.sql(f"""
WITH march AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_31d,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_31d,

        SUM(
            gsc_avg_position * gsc_impressions
        ) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS avg_position_31d,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS active_gsc_days

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_impressions,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days

    FROM {APRIL}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions_31d,
    m.clicks_31d,

    100.0 * m.clicks_31d
        / NULLIF(m.impressions_31d, 0) AS ctr_31d,

    m.avg_position_31d,
    m.active_gsc_days,

    (
        a.april_impressions
        < 0.80 * m.impressions_31d
    ) AS declined_next_month

FROM march AS m

INNER JOIN april AS a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE
    m.impressions_31d >= 100
    AND m.active_gsc_days > 0
    AND a.april_gsc_days > 0
    AND m.impressions_31d IS NOT NULL
    AND m.avg_position_31d IS NOT NULL
    AND m.avg_position_31d > 0
    AND a.april_impressions IS NOT NULL
""").df()

print("\nFeature frame ready.")
print("Rows:", len(feature_frame))
print("Clients:", feature_frame[GROUP].nunique())


# ------------------------------------------------------------
# 3. Reproduce the validated grouped holdout
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        feature_frame,
        feature_frame[TARGET],
        groups=feature_frame[GROUP]
    )
)

train_df = feature_frame.iloc[train_idx].copy()
test_df = feature_frame.iloc[test_idx].copy()

print("\nGrouped split")
print("-" * 50)
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print(
    "Client overlap:",
    len(
        set(train_df[GROUP])
        .intersection(set(test_df[GROUP]))
    )
)


# ------------------------------------------------------------
# 4. Train the same Logistic Regression model
# ------------------------------------------------------------

model = Pipeline([
    (
        "standard_scaler",
        StandardScaler()
    ),
    (
        "logistic_regression",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

model.fit(
    train_df[FEATURES],
    train_df[TARGET]
)

test_df["risk_score"] = model.predict_proba(
    test_df[FEATURES]
)[:, 1]


# ------------------------------------------------------------
# 5. Build a position-aware CTR reference from TRAINING data
# ------------------------------------------------------------

position_bins = [
    0,
    3,
    10,
    20,
    50,
    np.inf
]

position_labels = [
    "<=3",
    "4-10",
    "11-20",
    "21-50",
    "51+"
]

train_df["position_bucket"] = pd.cut(
    train_df["avg_position_31d"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

test_df["position_bucket"] = pd.cut(
    test_df["avg_position_31d"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

ctr_reference = (
    train_df
    .groupby(
        "position_bucket",
        observed=False
    )["ctr_31d"]
    .median()
)

test_df["expected_ctr_for_position"] = (
    test_df["position_bucket"]
    .map(ctr_reference)
    .astype(float)
)

test_df["ctr_gap"] = (
    test_df["ctr_31d"]
    - test_df["expected_ctr_for_position"]
)

high_visibility_cutoff = (
    train_df["impressions_31d"]
    .quantile(0.75)
)


# ------------------------------------------------------------
# 6. Rank the held-out content
# ------------------------------------------------------------

ranked = (
    test_df
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked["rank"] = (
    np.arange(len(ranked)) + 1
)

top20 = ranked.head(20).copy()


# ------------------------------------------------------------
# 7. Convert model ranking into human-readable archetypes,
#    reason codes and recommended actions
# ------------------------------------------------------------

def assign_playbook_action(row):

    low_ctr = (
        row["ctr_gap"] < 0
    )

    high_visibility = (
        row["impressions_31d"]
        >= high_visibility_cutoff
    )

    if low_ctr and high_visibility:
        return pd.Series({
            "archetype":
                "HIGH_VISIBILITY_LOW_CTR",

            "reason_code":
                "LOW_CTR_HIGH_VISIBILITY",

            "recommended_action":
                "REVIEW_SNIPPET_INTENT_AND_FRESHNESS",

            "cost_value":
                "High value / medium effort"
        })

    elif low_ctr:
        return pd.Series({
            "archetype":
                "LOW_CTR_FOR_POSITION",

            "reason_code":
                "LOW_CTR_FOR_POSITION",

            "recommended_action":
                "REVIEW_TITLE_SNIPPET_AND_INTENT",

            "cost_value":
                "Medium value / low-medium effort"
        })

    elif high_visibility:
        return pd.Series({
            "archetype":
                "HIGH_VISIBILITY_RISK",

            "reason_code":
                "HIGH_VOLUME_OPPORTUNITY",

            "recommended_action":
                "CHECK_FRESHNESS_RELEVANCE_AND_COMPETITION",

            "cost_value":
                "High value / medium effort"
        })

    else:
        return pd.Series({
            "archetype":
                "GENERAL_RISK_WATCHLIST",

            "reason_code":
                "MODEL_RISK_WATCHLIST",

            "recommended_action":
                "MANUAL_DIAGNOSTIC_REVIEW",

            "cost_value":
                "Medium value / low effort"
        })


action_fields = top20.apply(
    assign_playbook_action,
    axis=1
)

top20 = pd.concat(
    [
        top20.reset_index(drop=True),
        action_fields.reset_index(drop=True)
    ],
    axis=1
)


# ------------------------------------------------------------
# 8. Add human-review and evidence notes
# ------------------------------------------------------------

top20["human_review_required"] = True

top20["confidence_note"] = (
    "Model ranking is directional decision-support, "
    "not certainty."
)

top20["theory_refresh_note"] = (
    "Freshness may be worth checking during review, "
    "but refresh evidence is observational and does "
    "not guarantee improvement."
)


# ------------------------------------------------------------
# 9. Retrospective validation check
#    The future label is used ONLY here to evaluate the queue.
# ------------------------------------------------------------

precision_at_20 = (
    top20[TARGET]
    .astype(int)
    .mean()
)

print("\nRanked action queue")
print("-" * 50)
print(
    "Retrospective Precision@20:",
    round(precision_at_20, 3)
)

print(
    "Correct future declines in top 20:",
    int(top20[TARGET].sum()),
    "of 20"
)


# ------------------------------------------------------------
# 10. Create the action queue that will later be exported
# ------------------------------------------------------------

action_queue = top20[[
    "rank",
    "content_hash_id",
    "risk_score",
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
    "archetype",
    "reason_code",
    "recommended_action",
    "cost_value",
    "human_review_required",
    "confidence_note",
    "theory_refresh_note"
]].copy()

action_queue["risk_score"] = (
    action_queue["risk_score"]
    .round(4)
)

action_queue["ctr_31d"] = (
    action_queue["ctr_31d"]
    .round(3)
)

action_queue["avg_position_31d"] = (
    action_queue["avg_position_31d"]
    .round(2)
)

display(action_queue)

Hugging Face token loaded successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature frame ready.
Rows: 100893
Clients: 43

Grouped split
--------------------------------------------------
Training rows: 85702
Test rows: 15191
Client overlap: 0

Ranked action queue
--------------------------------------------------
Retrospective Precision@20: 0.8
Correct future declines in top 20: 16 of 20


,rank,content_hash_id,risk_score,impressions_31d,ctr_31d,avg_position_31d,active_gsc_days,archetype,reason_code,recommended_action,cost_value,human_review_required,confidence_note,theory_refresh_note
0,1,content_e3a63930f7422987,0.6729,447.0,0.0,2.03,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
1,2,content_e4b8aa114663611b,0.6728,206.0,0.0,2.89,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
2,3,content_db9d5ae0c17b7460,0.6724,551.0,0.0,1.98,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
3,4,content_4a25999c6672a162,0.6711,430.0,0.0,3.22,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
4,5,content_9eeba1f980fd7436,0.6711,587.0,0.0,2.75,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
5,6,content_93070561f9b163c3,0.6710,645.0,0.0,2.59,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
6,7,content_26ad8c3d6d0e6755,0.6709,320.0,0.0,3.72,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
7,8,content_56467aab5997e039,0.6707,231.0,0.0,4.17,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
8,9,content_5e9ca17e3735e391,0.6706,462.0,0.0,3.46,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."
9,10,content_e47b91d6e5daef37,0.6705,143.0,0.0,4.61,31,LOW_CTR_FOR_POSITION,LOW_CTR_FOR_POSITION,REVIEW_TITLE_SNIPPET_AND_INTENT,Medium value / low-medium effort,True,"Model ranking is directional decision-support,...","Freshness may be worth checking during review,..."


## 2. Intended use and limits

### Intended use

This playbook is intended for content analysts, SEO specialists, editors, or other reviewers who need to decide which content items should be inspected first.

The model uses search-performance information available at the end of March 2026 to rank content by estimated risk of a substantial impression decline in April. The ranked queue helps a reviewer allocate limited attention to higher-risk items first.

The output should be used for:

- prioritizing content for manual review;
- identifying high-risk items that may deserve attention;
- suggesting diagnostic checks such as CTR, title/snippet alignment, search intent, relevance, freshness, and competition;
- supporting decisions about where limited review effort may provide the most value.

### Limits

The model does not determine why a page declined and does not prove that changing any particular content element will improve performance.

Its validated performance is directional rather than certain. The original grouped holdout achieved Precision@20 of 0.80, while stronger grouped cross-validation produced a mean Precision@20 of 0.71. Performance therefore varies across unseen client groups.

The playbook is only appropriate when the incoming data follows the same basic definitions and feature construction used during training. Recommendations become less trustworthy when search behaviour, data coverage, client mix, measurement rules, or content conditions change substantially.

The model should not be used to automatically publish, delete, rewrite, redirect, or otherwise modify content. Final decisions require human review and relevant business or editorial context.

In [2]:
# ============================================================
# ML-10 — Section 2
# Intended use and limits
# ============================================================

intended_use_limits = pd.DataFrame({
    "area": [
        "Primary user",
        "Primary purpose",
        "Decision supported",
        "Prediction horizon",
        "Validated evidence",
        "Allowed interpretation",
        "Important limitation",
        "Out-of-scope use",
    ],

    "playbook_statement": [
        "Content analyst, SEO specialist, editor, or reviewer",

        "Prioritize which content items should be manually reviewed first",

        "Review priority and diagnostic investigation",

        "March search-performance features used to rank risk of April decline",

        "Grouped holdout Precision@20 = 0.80; "
        "5-fold grouped validation mean Precision@20 = 0.71",

        "Directional decision-support ranking",

        "The model does not identify the cause of decline "
        "and performance can vary across unseen client groups",

        "Do not automatically publish, rewrite, delete, "
        "redirect, or otherwise modify content"
    ]
})

display(intended_use_limits)


# ------------------------------------------------------------
# Basic playbook checks
# ------------------------------------------------------------

assert len(action_queue) == 20, (
    "The ranked review queue should contain 20 items."
)

assert action_queue["human_review_required"].all(), (
    "Every recommendation must require human review."
)

print("\nPlaybook scope checks passed.")
print("Ranked items available for review:", len(action_queue))
print(
    "Human review required for all items:",
    action_queue["human_review_required"].all()
)

print(
    "\nInterpretation: use the model to prioritize review, "
    "not to make automatic content decisions."
)

,area,playbook_statement
0,Primary user,"Content analyst, SEO specialist, editor, or re..."
1,Primary purpose,Prioritize which content items should be manua...
2,Decision supported,Review priority and diagnostic investigation
3,Prediction horizon,March search-performance features used to rank...
4,Validated evidence,Grouped holdout Precision@20 = 0.80; 5-fold gr...
5,Allowed interpretation,Directional decision-support ranking
6,Important limitation,The model does not identify the cause of decli...
7,Out-of-scope use,"Do not automatically publish, rewrite, delete,..."



Playbook scope checks passed.
Ranked items available for review: 20
Human review required for all items: True

Interpretation: use the model to prioritize review, not to make automatic content decisions.


## 3. Human review + the no-go list

### Human review rules

Every item in the ranked queue must be reviewed by a person before any content change is made.

For each recommendation, the reviewer should check:

1. **Search intent** — whether the page still matches what users appear to be looking for.
2. **Title and snippet** — whether the search result accurately and clearly represents the page.
3. **CTR context** — whether low CTR is genuinely unusual for the page's search position.
4. **Content relevance and quality** — whether important information is missing, outdated, unclear, or poorly structured.
5. **Freshness** — whether the subject actually requires updated information. Freshness is a diagnostic hypothesis, not an automatic fix.
6. **Recent search changes** — whether seasonality, competition, ranking changes, or changes in demand could explain the signal.
7. **Business/editorial context** — whether there are reasons outside the model that make a change inappropriate.

The model score decides review priority only. It does not decide what edit should be made.

### No-go list

The following actions must never be performed automatically from the model score alone:

- automatically rewrite page content;
- automatically change titles or metadata;
- automatically publish generated content;
- automatically delete or unpublish a page;
- automatically redirect URLs;
- automatically change canonical or indexing settings;
- automatically make claims about why traffic declined;
- automatically refresh content simply because freshness is suspected;
- automatically treat a high-risk score as proof that a page will decline.

A reviewer may decide that **no action is appropriate** after examining the page and its context. This is expected because the validation showed false positives as well as correct rankings.

In [3]:
# ============================================================
# ML-10 — Section 3
# Human review + the no-go list
# ============================================================

# ------------------------------------------------------------
# 1. Human review checklist
# ------------------------------------------------------------

human_review_rules = pd.DataFrame({
    "review_check": [
        "Search intent",
        "Title and snippet",
        "CTR context",
        "Content relevance and quality",
        "Freshness",
        "Recent search changes",
        "Business/editorial context",
    ],

    "reviewer_question": [
        "Does the page still match the likely search intent?",
        "Do the title and snippet clearly represent the page?",
        "Is the CTR actually weak relative to the page's position?",
        "Is important information missing, outdated, unclear, or poorly structured?",
        "Does this topic genuinely require updated information?",
        "Could seasonality, competition, ranking movement, or demand explain the signal?",
        "Is there business or editorial context the model cannot see?",
    ]
})

print("Human review checklist")
display(human_review_rules)


# ------------------------------------------------------------
# 2. Actions that must NOT be automated
# ------------------------------------------------------------

no_go_list = pd.DataFrame({
    "no_go_action": [
        "AUTO_REWRITE_CONTENT",
        "AUTO_CHANGE_TITLE_METADATA",
        "AUTO_PUBLISH",
        "AUTO_DELETE_OR_UNPUBLISH",
        "AUTO_REDIRECT",
        "AUTO_CHANGE_INDEXING",
        "AUTO_ASSIGN_CAUSE",
        "AUTO_REFRESH",
        "AUTO_TREAT_SCORE_AS_CERTAINTY",
    ],

    "reason": [
        "The model ranks review risk; it does not determine the correct rewrite.",
        "A title or metadata change requires intent and editorial review.",
        "Generated or edited content requires human approval before publication.",
        "A risk score is not sufficient evidence to remove content.",
        "Redirect decisions require technical and business context.",
        "Indexing decisions can have high-impact SEO consequences.",
        "The observational model cannot establish why performance changed.",
        "Freshness evidence is observational and does not prove refreshing will help.",
        "Validation includes false positives, so a high score is not certainty.",
    ]
})

print("\nNo-go list")
display(no_go_list)


# ------------------------------------------------------------
# 3. Add governance fields to the ranked queue
# ------------------------------------------------------------

action_queue["automation_allowed"] = False
action_queue["review_status"] = "PENDING_HUMAN_REVIEW"

action_queue["decision_rule"] = (
    "Review first; change content only if human investigation "
    "finds sufficient evidence."
)


# ------------------------------------------------------------
# 4. Safety / governance checks
# ------------------------------------------------------------

assert action_queue["human_review_required"].all(), (
    "Every ranked recommendation must require human review."
)

assert (~action_queue["automation_allowed"]).all(), (
    "No ranked recommendation may permit automatic execution."
)

assert (
    action_queue["review_status"]
    == "PENDING_HUMAN_REVIEW"
).all(), (
    "Every item should begin in human-review status."
)


print("\nGovernance checks passed.")
print(
    "Items requiring human review:",
    int(action_queue["human_review_required"].sum()),
    "of",
    len(action_queue)
)

print(
    "Items allowed automatic execution:",
    int(action_queue["automation_allowed"].sum())
)

print(
    "\nNo-go rule: model scores prioritize investigation; "
    "they never authorize an automatic content change."
)

Human review checklist


,review_check,reviewer_question
0,Search intent,Does the page still match the likely search in...
1,Title and snippet,Do the title and snippet clearly represent the...
2,CTR context,Is the CTR actually weak relative to the page'...
3,Content relevance and quality,"Is important information missing, outdated, un..."
4,Freshness,Does this topic genuinely require updated info...
5,Recent search changes,"Could seasonality, competition, ranking moveme..."
6,Business/editorial context,Is there business or editorial context the mod...



No-go list


,no_go_action,reason
0,AUTO_REWRITE_CONTENT,The model ranks review risk; it does not deter...
1,AUTO_CHANGE_TITLE_METADATA,A title or metadata change requires intent and...
2,AUTO_PUBLISH,Generated or edited content requires human app...
3,AUTO_DELETE_OR_UNPUBLISH,A risk score is not sufficient evidence to rem...
4,AUTO_REDIRECT,Redirect decisions require technical and busin...
5,AUTO_CHANGE_INDEXING,Indexing decisions can have high-impact SEO co...
6,AUTO_ASSIGN_CAUSE,The observational model cannot establish why p...
7,AUTO_REFRESH,Freshness evidence is observational and does n...
8,AUTO_TREAT_SCORE_AS_CERTAINTY,"Validation includes false positives, so a high..."



Governance checks passed.
Items requiring human review: 20 of 20
Items allowed automatic execution: 0

No-go rule: model scores prioritize investigation; they never authorize an automatic content change.


## 4. Monitoring / retrain triggers

### Monitoring and retraining

The model should not be assumed to remain reliable forever. Search behaviour, client mix, content characteristics, data availability, and ranking patterns can change over time.

I would monitor both model performance and incoming feature distributions.

The model should be reviewed or retrained when one or more of the following occurs:

- **Precision@20 drops below 0.60** on newly observed outcomes.
- Precision@20 falls substantially below the validated grouped cross-validation result of 0.71.
- **CTR, impressions, average position, or active GSC days shift materially** from the training-period distributions.
- Missing or unavailable GSC data increases substantially.
- The mix of clients or content becomes meaningfully different from the training data.
- The definition of the outcome, features, data source, or measurement window changes.
- Reviewers repeatedly reject high-ranked recommendations as irrelevant or incorrect.
- A major search-platform or measurement change makes the historical relationships less representative.

A trigger does not automatically retrain the model. It starts a human investigation first. The reviewer should check data quality, feature definitions, validation performance, and business context before deciding whether retraining is necessary.

In [4]:
# ============================================================
# ML-10 — Section 4
# Monitoring / retrain triggers
# ============================================================

# ------------------------------------------------------------
# 1. Define practical monitoring rules
# ------------------------------------------------------------

monitoring_triggers = pd.DataFrame({
    "monitor": [
        "Precision@20",
        "Precision@20 vs validated result",
        "Feature distribution drift",
        "GSC data availability",
        "Client/content mix",
        "Feature or label definition",
        "Human reviewer rejection rate",
        "External search environment",
    ],

    "trigger": [
        "New-outcome Precision@20 < 0.60",
        "Performance materially below grouped CV mean of 0.71",
        "Large shift in CTR, impressions, position, or active days",
        "Material increase in missing/unavailable search data",
        "New data population differs strongly from training population",
        "Any change to feature, label, source, or measurement definition",
        "Reviewers frequently reject high-ranked recommendations",
        "Major platform or measurement change",
    ],

    "response": [
        "Audit recent predictions and consider retraining",
        "Revalidate across client groups before continued use",
        "Investigate drift and compare with training distributions",
        "Check data pipeline before trusting model scores",
        "Run grouped validation on the new population",
        "Rebuild and revalidate the model before use",
        "Review failure cases and recommendation logic",
        "Reassess whether historical patterns remain representative",
    ]
})

display(monitoring_triggers)


# ------------------------------------------------------------
# 2. Record reference performance from previous validation
# ------------------------------------------------------------

monitoring_reference = {
    "grouped_holdout_precision_at_20": 0.80,
    "grouped_cv_mean_precision_at_20": 0.71,
    "precision_at_20_review_threshold": 0.60,
}

print("\nMonitoring reference")
print("-" * 50)

for metric, value in monitoring_reference.items():
    print(f"{metric}: {value}")


# ------------------------------------------------------------
# 3. Simple trigger function for future monitoring
# ------------------------------------------------------------

def check_precision_trigger(current_precision_at_20):
    threshold = monitoring_reference[
        "precision_at_20_review_threshold"
    ]

    if current_precision_at_20 < threshold:
        return "TRIGGER_REVIEW"

    return "CONTINUE_MONITORING"


# Example using the current held-out result
current_status = check_precision_trigger(
    precision_at_20
)

print("\nCurrent example status:", current_status)


# ------------------------------------------------------------
# 4. Governance rule
# ------------------------------------------------------------

print(
    "\nRetraining rule: a monitoring trigger starts a human "
    "investigation; it does not automatically retrain or deploy a model."
)

,monitor,trigger,response
0,Precision@20,New-outcome Precision@20 < 0.60,Audit recent predictions and consider retraining
1,Precision@20 vs validated result,Performance materially below grouped CV mean o...,Revalidate across client groups before continu...
2,Feature distribution drift,"Large shift in CTR, impressions, position, or ...",Investigate drift and compare with training di...
3,GSC data availability,Material increase in missing/unavailable searc...,Check data pipeline before trusting model scores
4,Client/content mix,New data population differs strongly from trai...,Run grouped validation on the new population
5,Feature or label definition,"Any change to feature, label, source, or measu...",Rebuild and revalidate the model before use
6,Human reviewer rejection rate,Reviewers frequently reject high-ranked recomm...,Review failure cases and recommendation logic
7,External search environment,Major platform or measurement change,Reassess whether historical patterns remain re...



Monitoring reference
--------------------------------------------------
grouped_holdout_precision_at_20: 0.8
grouped_cv_mean_precision_at_20: 0.71
precision_at_20_review_threshold: 0.6

Current example status: CONTINUE_MONITORING

Retraining rule: a monitoring trigger starts a human investigation; it does not automatically retrain or deploy a model.


## 5. Exports for the paper

### Paper exports

This notebook exports the final ranked review queue to `work/outputs/action_queue.csv`.

The queue contains only the fields needed for the action playbook. It does not include client names, URLs, private queries, or the future outcome label used for retrospective evaluation.

I also export a small JSON file containing the main validation and governance metrics. These files give the research paper reproducible evidence for the recommendation section.

The CSV is a generated data artifact and can be recreated by running this notebook. The exported metrics record the exact numbers used when describing the playbook.

In [5]:
# ============================================================
# ML-10 — Section 5
# Exports for the paper
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Create export directory
# ------------------------------------------------------------

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Export the ranked action queue
# ------------------------------------------------------------

queue_path = OUTPUT_DIR / "action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)


# ------------------------------------------------------------
# 3. Export the small metrics receipt
# ------------------------------------------------------------

metrics = {
    "model": "Logistic Regression",

    "target": "declined_next_month",

    "features": FEATURES,

    "ranked_queue_size": int(
        len(action_queue)
    ),

    "grouped_holdout_precision_at_20": round(
        float(precision_at_20),
        3
    ),

    "grouped_cv_mean_precision_at_20": 0.71,

    "precision_at_20_review_threshold": 0.60,

    "correct_future_declines_in_top_20": int(
        top20[TARGET].sum()
    ),

    "human_review_required_for_all": bool(
        action_queue[
            "human_review_required"
        ].all()
    ),

    "automatic_execution_allowed": bool(
        action_queue[
            "automation_allowed"
        ].any()
    ),

    "claim_level":
        "directional decision-support",

    "retraining_policy":
        "monitoring triggers require human investigation before retraining"
}

metrics_path = (
    OUTPUT_DIR
    / "action_playbook_metrics.json"
)

with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 4. Verify the exports
# ------------------------------------------------------------

assert queue_path.exists(), (
    "action_queue.csv was not created."
)

assert metrics_path.exists(), (
    "action_playbook_metrics.json was not created."
)

assert len(action_queue) == 20, (
    "Expected 20 rows in the exported queue."
)

assert not action_queue[
    "content_hash_id"
].isna().any(), (
    "The export contains missing content IDs."
)

assert "declined_next_month" not in (
    action_queue.columns
), (
    "Future outcome label should not be "
    "included in the operational queue."
)


print("Paper exports created successfully.")
print("-" * 50)
print("Queue:", queue_path)
print("Metrics:", metrics_path)

print(
    "\nRows exported:",
    len(action_queue)
)

print(
    "Columns exported:",
    len(action_queue.columns)
)

print(
    "Future outcome included in queue:",
    "declined_next_month"
    in action_queue.columns
)

print(
    "\nMain paper metric:"
)

print(
    "Grouped holdout Precision@20 =",
    metrics[
        "grouped_holdout_precision_at_20"
    ]
)

print(
    "Grouped CV mean Precision@20 =",
    metrics[
        "grouped_cv_mean_precision_at_20"
    ]
)

print(
    "\nExport check passed."
)

Paper exports created successfully.
--------------------------------------------------
Queue: work/outputs/action_queue.csv
Metrics: work/outputs/action_playbook_metrics.json

Rows exported: 20
Columns exported: 17
Future outcome included in queue: False

Main paper metric:
Grouped holdout Precision@20 = 0.8
Grouped CV mean Precision@20 = 0.71

Export check passed.
